In [20]:
import polars as pl
import psycopg2

In [21]:
df = pl.read_parquet("./data/active_genes.parquet")
# Connect to petadex
DB = {
    "host":     "petadex.ccz9y6yshbls.us-east-1.rds.amazonaws.com",
    "port":     5432,
    "database": "petadex",
    "user":     "readonly_user",
    "password": "petadex",
}

conn = psycopg2.connect(**DB)
cur  = conn.cursor()

In [22]:
cur.execute("""
    SELECT * 
    FROM sra_metadata
    WHERE acc IN %s
""", (tuple(df["gene"]),)
)
rows = cur.fetchall()
columns = [desc[0] for desc in cur.description]
query_df = pl.DataFrame(rows, schema=columns, orient="row")

In [23]:
# join the metadata to the active genes
df = df.join(query_df, left_on="gene", right_on="acc", how="left")

In [ ]:
# drop rows where lat and lon are null, since we can't use those for clustering
df = df.filter(
    (pl.col('latitude').is_not_null()) & (pl.col('longitude').is_not_null())
)

In [29]:
df.write_parquet("./data/active_samples_metadata.parquet")